In [1]:
# ============================================================================
# STAGE 2 -- DAG extraction (Type B: Planning / Fact / Reasoning / Conclusion)
#
# STANDALONE: this cell does its own cache redirection, imports, and config.
# It only needs the Stage 1 eval CSVs to exist on disk under STAGE1_OUTPUT_DIR.
#
# Reads every *_eval_cots.csv (holdout splits skipped), feeds each reasoning
# trace to Qwen2.5-32B-Instruct, parses a Type-B DAG following the paper's
# bundle-based premise encoding, validates, and writes one JSONL per dataset
# with one DAG per row.
#
# IMPORTANT -- before running this cell, set these in the SHELL that launches
# Jupyter (NOT in a cell, because by then transformers/vllm have cached the
# default paths at import time):
#
#   export HF_HOME=/work/hdd/bfrc/hf
#   export HF_HUB_CACHE=/work/hdd/bfrc/hf/hub
#   export TRANSFORMERS_CACHE=/work/hdd/bfrc/hf
#   export HF_DATASETS_CACHE=/work/hdd/bfrc/hf/datasets
#   export VLLM_CACHE_ROOT=/work/hdd/bfrc/vllm
#   export TRITON_CACHE_DIR=/work/hdd/bfrc/triton
#   export TORCHINDUCTOR_CACHE_DIR=/work/hdd/bfrc/torch_inductor
#   export TMPDIR=/work/hdd/bfrc/tmp
#
# This cell ALSO sets them defensively, AND passes download_dir directly to
# vLLM so the weights go to /work/hdd/bfrc no matter what.
# ============================================================================

# ----------------------------------------------------------------------------
# CACHE REDIRECTION + SANITY CHECK
# Set env vars first, then verify huggingface_hub resolved to the right path.
# If not, abort -- don't download 65 GB to the wrong place.
# ----------------------------------------------------------------------------
import os

os.environ["VLLM_USE_DEEP_GEMM"] = "0"
os.environ["VLLM_MOE_USE_DEEP_GEMM"] = "0"
os.environ["VLLM_DEEP_GEMM_WARMUP"] = "skip"

HF_CACHE_ROOT = "/work/hdd/bfrc"
os.environ["HF_HOME"]                 = f"{HF_CACHE_ROOT}/hf"
os.environ["HF_HUB_CACHE"]            = f"{HF_CACHE_ROOT}/hf/hub"
os.environ["TRANSFORMERS_CACHE"]      = f"{HF_CACHE_ROOT}/hf"
os.environ["HF_DATASETS_CACHE"]       = f"{HF_CACHE_ROOT}/hf/datasets"
os.environ["VLLM_CACHE_ROOT"]         = f"{HF_CACHE_ROOT}/vllm"
os.environ["TRITON_CACHE_DIR"]        = f"{HF_CACHE_ROOT}/triton"
os.environ["TORCHINDUCTOR_CACHE_DIR"] = f"{HF_CACHE_ROOT}/torch_inductor"
os.environ["TMPDIR"]                  = f"{HF_CACHE_ROOT}/tmp"
for p in (os.environ["HF_HOME"], os.environ["HF_HUB_CACHE"],
          os.environ["HF_DATASETS_CACHE"], os.environ["VLLM_CACHE_ROOT"],
          os.environ["TRITON_CACHE_DIR"], os.environ["TORCHINDUCTOR_CACHE_DIR"],
          os.environ["TMPDIR"]):
    os.makedirs(p, exist_ok=True)

# Sanity-check what huggingface_hub actually resolved. If anything was already
# imported earlier in the kernel (e.g. by a previous cell), the env vars set
# above came too late and the path is already pinned to ~/.cache/huggingface.
import importlib, sys
if "huggingface_hub" in sys.modules:
    importlib.reload(sys.modules["huggingface_hub"])
    if "huggingface_hub.constants" in sys.modules:
        importlib.reload(sys.modules["huggingface_hub.constants"])
from huggingface_hub import constants as _hf_constants
_resolved_hub_cache = str(_hf_constants.HF_HUB_CACHE)
print(f"[cache check] huggingface_hub resolved HF_HUB_CACHE = {_resolved_hub_cache}")
if not _resolved_hub_cache.startswith(HF_CACHE_ROOT):
    print(f"[cache check] WARNING: huggingface_hub is pointing OUTSIDE "
          f"{HF_CACHE_ROOT}. We will force-override via vLLM's download_dir "
          f"argument below, but if you also see partial downloads in "
          f"~/.cache/huggingface, restart your kernel with the env vars "
          f"set BEFORE launching Jupyter.")
else:
    print(f"[cache check] OK -- huggingface_hub will write under {HF_CACHE_ROOT}")

# ----------------------------------------------------------------------------
# Now the real imports.
# ----------------------------------------------------------------------------
import gc, glob, json, re, time
from typing import Dict, List, Optional, Set, Tuple

import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
# Where Stage 1 wrote its CSVs:
STAGE1_OUTPUT_DIR = "./outputs_2234"

# Where this cell writes its DAGs:
DAG_OUTPUT_DIR    = "./outputs_2234"
os.makedirs(DAG_OUTPUT_DIR, exist_ok=True)

# Where vLLM should download the extractor weights. This is the BULLETPROOF
# path -- vLLM honors this regardless of env vars.
VLLM_DOWNLOAD_DIR = f"{HF_CACHE_ROOT}/hf/hub"
os.makedirs(VLLM_DOWNLOAD_DIR, exist_ok=True)

# Only the eval splits.
EVAL_CSV_GLOB = os.path.join(STAGE1_OUTPUT_DIR, "*_eval_cots.csv")

# Extractor model + sampling. Temperature 0.3 per paper, deterministic seed.
EXTRACTOR_MODEL    = "Qwen/Qwen2.5-32B-Instruct"
EXTRACTOR_TEMP     = 0.3
EXTRACTOR_TOP_P    = 0.95
EXTRACTOR_MAX_TOK  = 2048      # JSON adds overhead vs raw text
EXTRACTOR_MAX_LEN  = 8192
EXTRACTOR_GPU_UTIL = 0.90
EXTRACTOR_TP       = 1         # tensor_parallel_size -- bump for multi-GPU

# ----------------------------------------------------------------------------
# Extraction prompt. Given the trace AND its final answer, the extractor's
# only job is to recover the supporting graph -- it does NOT decide the
# verdict. The Conclusion node must express the supplied final answer.
# ----------------------------------------------------------------------------
EXTRACTOR_SYSTEM = (
    "You are a structured-reasoning extractor. Given a reasoning trace and "
    "its final answer, you produce a Directed Acyclic Graph (DAG) in JSON "
    "that captures the trace's logical structure. You output ONLY a single "
    "JSON object inside <dag>...</dag> tags. No prose. No markdown fences."
)


def build_extractor_prompt(reasoning: str, final_answer: str,
                           question_blob: str) -> str:
    return f"""TASK: Extract a Type-B reasoning DAG from this trace.

NODE TYPES (use exactly these strings):
  - "Planning":   meta-reasoning, problem decomposition, deciding what to check
  - "Fact":       factual statements drawn from the question/statute/scenario
  - "Reasoning":  intermediate inferential steps linking facts to outcome
  - "Conclusion": the final determination

REQUIRED: the DAG must contain exactly ONE node of type "Conclusion", whose
`text` expresses the final answer below (paraphrase it however the trace did,
but it must be the same verdict). All other nodes must transitively support
this Conclusion.

BUNDLE STRUCTURE: each node has
  - "id":      unique short string id (e.g. "p1", "f1", "r1", "c1")
  - "type":    one of the four node types above
  - "text":    short natural-language description (one sentence)
  - "support": list of bundles. Each bundle is a list of node ids that
                 JOINTLY (conjunctive AND) justify this node. Different
                 bundles in the list are alternative (disjunctive OR)
                 independent justifications.
  - "defeats": list of bundles whose joint truth would SUPPRESS this node.
                 Use [] if the trace mentions no counter-considerations.
                 Do NOT invent defeaters.

RULES:
  - Facts may have empty support (they are atomic premises from the question).
  - Planning/Reasoning/Conclusion nodes MUST have >=1 support bundle.
  - Every id referenced in a bundle must exist as a node in the graph.
  - The graph must be acyclic.
  - Do NOT add information the trace does not contain.

----------------------------------------
QUESTION CONTEXT:
{question_blob}

FINAL ANSWER (must be the Conclusion node's verdict):
{final_answer}

REASONING TRACE:
{reasoning}
----------------------------------------

Output ONLY this:
<dag>{{"nodes": [
  {{"id": "...", "type": "Planning|Fact|Reasoning|Conclusion", "text": "...",
    "support": [["id1","id2"], ["id3"]], "defeats": []}}
]}}</dag>"""


def question_blob_for(row: Dict) -> str:
    """Compact question summary so the extractor can resolve references like
    'the hypothesis' or 'option B' that appear inside the trace."""
    ds = row.get("dataset", "")
    if ds == "sara":
        return (f"STATUTE: {row.get('statute') or '(see case)'}\n"
                f"CASE: {row.get('case', '')}\n"
                f"HYPOTHESIS: {row.get('hypothesis', '')}")
    if ds == "argkp":
        stance = str(row.get("stance", ""))
        stance_word = "PRO" if stance in ("1", "+1") else "CON"
        return f"TOPIC: {row.get('topic', '')}\nSTANCE: {stance_word}"
    if ds == "gpqa":
        return (f"QUESTION: {row.get('question_text', '')}\n"
                f"A. {row.get('option_a', '')}\n"
                f"B. {row.get('option_b', '')}\n"
                f"C. {row.get('option_c', '')}\n"
                f"D. {row.get('option_d', '')}")
    return ""


# ----------------------------------------------------------------------------
# Extract the <dag>...</dag> JSON, parse, and run validation. Returns a dict
# {"dag": parsed_dict_or_None, "parse_ok": bool, "errors": [list of strings]}.
# parse_ok=True means we have a syntactically valid, validated DAG. Failures
# are still written so you can retry / filter / inspect.
# ----------------------------------------------------------------------------
_DAG_TAG_RE = re.compile(r"<dag>(.*?)</dag>", re.DOTALL | re.IGNORECASE)
VALID_TYPES = {"Planning", "Fact", "Reasoning", "Conclusion"}


def _preds(node: Dict) -> List[str]:
    out = []
    for bundle in node.get("support", []):
        if isinstance(bundle, list):
            out.extend(bundle)
    for bundle in node.get("defeats", []):
        if isinstance(bundle, list):
            out.extend(bundle)
    return out


def parse_and_validate_dag(raw: str) -> Dict:
    errors: List[str] = []

    # 1. Extract the JSON blob, with a fallback for models that forget the tags.
    m = _DAG_TAG_RE.search(raw)
    if m:
        blob = m.group(1).strip()
    else:
        try:
            blob = raw[raw.index("{"): raw.rindex("}") + 1]
        except ValueError:
            return {"dag": None, "parse_ok": False,
                    "errors": ["no <dag> tag and no { ... } block found"]}

    try:
        dag = json.loads(blob)
    except json.JSONDecodeError as e:
        return {"dag": None, "parse_ok": False,
                "errors": [f"json decode failed: {e}"]}

    if not isinstance(dag, dict) or "nodes" not in dag:
        return {"dag": dag, "parse_ok": False,
                "errors": ["top-level object missing 'nodes' key"]}

    nodes = dag["nodes"]
    if not isinstance(nodes, list) or not nodes:
        return {"dag": dag, "parse_ok": False,
                "errors": ["'nodes' must be a non-empty list"]}

    # 2. Per-node structural checks: id uniqueness, type, required fields.
    id_to_node: Dict[str, Dict] = {}
    for i, n in enumerate(nodes):
        if not isinstance(n, dict):
            errors.append(f"node[{i}] not a dict")
            continue
        nid = n.get("id")
        ntype = n.get("type")
        if not isinstance(nid, str) or not nid:
            errors.append(f"node[{i}] missing/invalid id")
            continue
        if nid in id_to_node:
            errors.append(f"duplicate node id: {nid}")
            continue
        if ntype not in VALID_TYPES:
            errors.append(f"node {nid}: invalid type {ntype!r}")
            continue
        if "text" not in n or not isinstance(n["text"], str):
            errors.append(f"node {nid}: missing/invalid text")
        for key in ("support", "defeats"):
            if key not in n:
                n[key] = []
            if not isinstance(n[key], list):
                errors.append(f"node {nid}: {key} must be a list")
                n[key] = []
        id_to_node[nid] = n

    if errors:
        return {"dag": dag, "parse_ok": False, "errors": errors}

    # 3. Bundle reference + cardinality checks.
    for nid, n in id_to_node.items():
        for bundle_list_name in ("support", "defeats"):
            for j, bundle in enumerate(n[bundle_list_name]):
                if not isinstance(bundle, list):
                    errors.append(f"node {nid}: {bundle_list_name}[{j}] not a list")
                    continue
                for ref in bundle:
                    if ref not in id_to_node:
                        errors.append(f"node {nid}: {bundle_list_name}[{j}] "
                                      f"references unknown id {ref!r}")
        if n["type"] != "Fact" and not n["support"]:
            errors.append(f"node {nid} of type {n['type']} has no support "
                          f"bundles (only Facts may be atomic)")

    # 4. Exactly one Conclusion node.
    concs = [nid for nid, n in id_to_node.items() if n["type"] == "Conclusion"]
    if len(concs) != 1:
        errors.append(f"expected exactly 1 Conclusion node, got {len(concs)}")

    # 5. Acyclic check. Skip if there are reference errors -- our DFS assumes
    # every predecessor id resolves, otherwise it'd KeyError.
    has_ref_errors = any("references unknown id" in e for e in errors)
    if not has_ref_errors:
        WHITE, GRAY, BLACK = 0, 1, 2
        color = {nid: WHITE for nid in id_to_node}

        def has_cycle(start: str) -> bool:
            stack = [(start, iter(_preds(id_to_node[start])))]
            color[start] = GRAY
            while stack:
                node, it = stack[-1]
                nxt = next(it, None)
                if nxt is None:
                    color[node] = BLACK
                    stack.pop()
                    continue
                if color.get(nxt, WHITE) == GRAY:
                    return True
                if color.get(nxt, WHITE) == WHITE:
                    color[nxt] = GRAY
                    stack.append((nxt, iter(_preds(id_to_node[nxt]))))
            return False

        for nid in id_to_node:
            if color[nid] == WHITE and has_cycle(nid):
                errors.append("graph contains a cycle")
                break

    # 6. Derive gate types from support cardinality (paper sec. 3.2).
    for nid, n in id_to_node.items():
        nsup = len(n["support"])
        n["gate"] = ("Atomic" if nsup == 0
                     else "And" if nsup == 1
                     else "Or")

    return {"dag": dag, "parse_ok": not errors, "errors": errors}


# ----------------------------------------------------------------------------
# Resume support: read existing JSONL, return set of done (qid, model, samp).
# ----------------------------------------------------------------------------
def already_extracted_keys(jsonl_path: str) -> Set[Tuple[str, str, int]]:
    done: Set[Tuple[str, str, int]] = set()
    if not os.path.exists(jsonl_path):
        return done
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                row = json.loads(line)
                done.add((row["question_id"], row["model"], int(row["sample_idx"])))
            except (json.JSONDecodeError, KeyError, ValueError):
                continue
    return done


# ----------------------------------------------------------------------------
# Build work list from the eval CSVs (resume-aware, skip empty answers/reasoning).
# ----------------------------------------------------------------------------
eval_csv_paths = sorted(glob.glob(EVAL_CSV_GLOB))
print(f"\n[stage 2] found {len(eval_csv_paths)} eval CSVs under {STAGE1_OUTPUT_DIR}:")
for p in eval_csv_paths:
    print(f"  {p}")
if not eval_csv_paths:
    raise RuntimeError(f"No eval CSVs at {EVAL_CSV_GLOB}. Did Stage 1 run?")

work_items: List[Dict] = []
for csv_path in eval_csv_paths:
    ds_name = os.path.basename(csv_path).split("_eval_cots.csv")[0]
    jsonl_path = os.path.join(DAG_OUTPUT_DIR, f"{ds_name}_eval_dags.jsonl")
    done = already_extracted_keys(jsonl_path)

    df = pd.read_csv(csv_path, dtype=str).fillna("")
    skipped_no_answer = skipped_no_reasoning = skipped_resume = 0
    for _, row in df.iterrows():
        try:
            sample_idx = int(row["sample_idx"])
        except (ValueError, KeyError):
            continue
        if (row["question_id"], row["model"], sample_idx) in done:
            skipped_resume += 1
            continue
        if not row.get("predicted_label", "").strip():
            skipped_no_answer += 1
            continue
        if not row.get("reasoning", "").strip():
            skipped_no_reasoning += 1
            continue
        work_items.append({
            "row": row.to_dict(),
            "jsonl_path": jsonl_path,
            "ds_name": ds_name,
        })
    print(f"  [{ds_name}] {len(df)} traces, "
          f"{skipped_resume} resumed-done, "
          f"{skipped_no_answer} no-answer, "
          f"{skipped_no_reasoning} no-reasoning")

print(f"\n[stage 2] {len(work_items)} traces queued for DAG extraction")

if not work_items:
    print("[stage 2] nothing to do; all eval traces already extracted.")
else:
    # ------------------------------------------------------------------
    # Load extractor and prepare prompts.
    # ------------------------------------------------------------------
    print(f"\n[stage 2] loading extractor {EXTRACTOR_MODEL}")
    print(f"           weights will be downloaded to: {VLLM_DOWNLOAD_DIR}")
    tokenizer = AutoTokenizer.from_pretrained(
        EXTRACTOR_MODEL,
        cache_dir=VLLM_DOWNLOAD_DIR,   # belt-and-suspenders: pin tokenizer too
        trust_remote_code=True,
    )
    llm = LLM(
        model=EXTRACTOR_MODEL,
        dtype="bfloat16",
        trust_remote_code=True,
        gpu_memory_utilization=EXTRACTOR_GPU_UTIL,
        max_model_len=EXTRACTOR_MAX_LEN,
        tensor_parallel_size=EXTRACTOR_TP,
        download_dir=VLLM_DOWNLOAD_DIR,  # <-- the bulletproof override
    )

    prompts, sampling_params = [], []
    for w in work_items:
        row = w["row"]
        user_msg = build_extractor_prompt(
            reasoning=row["reasoning"],
            final_answer=row["predicted_label"],
            question_blob=question_blob_for(row),
        )
        messages = [{"role": "system", "content": EXTRACTOR_SYSTEM},
                    {"role": "user",   "content": user_msg}]
        prompt_str = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False)
        prompts.append(prompt_str)
        sampling_params.append(SamplingParams(
            temperature=EXTRACTOR_TEMP,
            top_p=EXTRACTOR_TOP_P,
            max_tokens=EXTRACTOR_MAX_TOK,
            seed=0,
            n=1,
        ))

    # ------------------------------------------------------------------
    # Dispatch as ONE big batched call. vLLM continuous-batches across this.
    # ------------------------------------------------------------------
    print(f"[stage 2] dispatching {len(prompts)} extractions to vLLM ...")
    t0 = time.time()
    outputs = llm.generate(prompts, sampling_params)
    wall = time.time() - t0
    print(f"[stage 2] done in {wall:.1f}s "
          f"({wall / max(1, len(prompts)):.2f}s/extraction avg)")

    # ------------------------------------------------------------------
    # Parse + validate + append JSONL. Files opened once each.
    # ------------------------------------------------------------------
    files: Dict[str, "io.TextIOBase"] = {}
    n_ok = n_fail = 0
    try:
        for w, out in zip(work_items, outputs):
            raw = out.outputs[0].text
            parsed = parse_and_validate_dag(raw)

            row = w["row"]
            record = {
                "question_id": row["question_id"],
                "dataset": row["dataset"],
                "split": row["split"],
                "model": row["model"],
                "sample_idx": int(row["sample_idx"]),
                "gold_label": row["gold_label"],
                "predicted_label": row["predicted_label"],
                "extractor_raw": raw,
                "dag": parsed["dag"],
                "parse_ok": parsed["parse_ok"],
                "errors": parsed["errors"],
            }
            if parsed["parse_ok"]:
                n_ok += 1
            else:
                n_fail += 1

            jsonl_path = w["jsonl_path"]
            if jsonl_path not in files:
                files[jsonl_path] = open(jsonl_path, "a", encoding="utf-8")
            files[jsonl_path].write(json.dumps(record, ensure_ascii=False) + "\n")
            files[jsonl_path].flush()
    finally:
        for f in files.values():
            f.close()

    print(f"[stage 2] extraction results: {n_ok} valid, {n_fail} failed validation")

    # Free GPU
    del llm, tokenizer, outputs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ----------------------------------------------------------------------------
# Summary table.
# ----------------------------------------------------------------------------
print("\n[stage 2] final tallies:")
for jsonl_path in sorted(glob.glob(os.path.join(DAG_OUTPUT_DIR, "*_eval_dags.jsonl"))):
    n_total = n_ok = 0
    by_model_ok: Dict[str, int] = {}
    by_model_total: Dict[str, int] = {}
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            n_total += 1
            m = rec.get("model", "?")
            by_model_total[m] = by_model_total.get(m, 0) + 1
            if rec.get("parse_ok"):
                n_ok += 1
                by_model_ok[m] = by_model_ok.get(m, 0) + 1
    pct = (100.0 * n_ok / n_total) if n_total else 0.0
    print(f"  {os.path.basename(jsonl_path)}: {n_ok}/{n_total} valid ({pct:.1f}%)")
    for m in sorted(by_model_total):
        ok = by_model_ok.get(m, 0)
        tot = by_model_total[m]
        print(f"    {m}: {ok}/{tot} valid ({100.0*ok/max(1,tot):.1f}%)")

[cache check] huggingface_hub resolved HF_HUB_CACHE = /work/hdd/bfrc/hf/hub
[cache check] OK -- huggingface_hub will write under /work/hdd/bfrc

[stage 2] found 4 eval CSVs under ./outputs_2234:
  ./outputs_2234/folio_eval_cots.csv
  ./outputs_2234/musr_mm_eval_cots.csv
  ./outputs_2234/musr_op_eval_cots.csv
  ./outputs_2234/musr_ta_eval_cots.csv
  [folio] 3060 traces, 0 resumed-done, 83 no-answer, 0 no-reasoning
  [musr_mm] 4000 traces, 0 resumed-done, 46 no-answer, 0 no-reasoning
  [musr_op] 4120 traces, 0 resumed-done, 7 no-answer, 0 no-reasoning
  [musr_ta] 4000 traces, 0 resumed-done, 22 no-answer, 0 no-reasoning

[stage 2] 15022 traces queued for DAG extraction

[stage 2] loading extractor Qwen/Qwen2.5-32B-Instruct
           weights will be downloaded to: /work/hdd/bfrc/hf/hub
INFO 05-23 13:31:12 [utils.py:233] non-default args: {'trust_remote_code': True, 'download_dir': '/work/hdd/bfrc/hf/hub', 'dtype': 'bfloat16', 'max_model_len': 8192, 'gpu_memory_utilization': 0.9, 'disable

Loading safetensors checkpoint shards:   0% Completed | 0/17 [00:00<?, ?it/s]


(EngineCore pid=3112104) INFO 05-23 13:31:36 [weight_utils.py:851] Prefetching checkpoint files: 10% (2/17)
(EngineCore pid=3112104) INFO 05-23 13:31:37 [weight_utils.py:851] Prefetching checkpoint files: 20% (4/17)
(EngineCore pid=3112104) INFO 05-23 13:31:40 [weight_utils.py:851] Prefetching checkpoint files: 30% (6/17)
(EngineCore pid=3112104) INFO 05-23 13:31:43 [weight_utils.py:851] Prefetching checkpoint files: 40% (7/17)
(EngineCore pid=3112104) INFO 05-23 13:31:44 [weight_utils.py:851] Prefetching checkpoint files: 50% (9/17)
(EngineCore pid=3112104) INFO 05-23 13:31:45 [weight_utils.py:851] Prefetching checkpoint files: 60% (11/17)
(EngineCore pid=3112104) INFO 05-23 13:31:45 [weight_utils.py:851] Prefetching checkpoint files: 70% (12/17)
(EngineCore pid=3112104) INFO 05-23 13:31:48 [weight_utils.py:851] Prefetching checkpoint files: 80% (14/17)
(EngineCore pid=3112104) INFO 05-23 13:31:50 [weight_utils.py:851] Prefetching checkpoint files: 90% (16/17)
(EngineCore pid=3112104)

(EngineCore pid=3112104) 2026-05-23 13:32:18,197 - INFO - autotuner.py:457 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=3112104) 2026-05-23 13:32:18,236 - INFO - autotuner.py:466 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 12.54it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:03<00:00, 15.47it/s]


(EngineCore pid=3112104) INFO 05-23 13:32:27 [gpu_model_runner.py:6133] Graph capturing finished in 9 secs, took 1.04 GiB
(EngineCore pid=3112104) INFO 05-23 13:32:27 [gpu_worker.py:599] CUDA graph pool memory: 1.04 GiB (actual), 0.9 GiB (estimated), difference: 0.13 GiB (13.0%).
(EngineCore pid=3112104) INFO 05-23 13:32:27 [core.py:299] init engine (profile, create kv cache, warmup model) took 34.91 s (compilation: 19.87 s)
(EngineCore pid=3112104) INFO 05-23 13:32:27 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
[stage 2] dispatching 15022 extractions to vLLM ...


Rendering prompts:   0%|          | 0/15022 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/15022 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/…

[stage 2] done in 3883.0s (0.26s/extraction avg)
(EngineCore pid=3112104) INFO 05-23 14:37:22 [core.py:1238] Shutdown initiated (timeout=0)
(EngineCore pid=3112104) INFO 05-23 14:37:22 [core.py:1261] Shutdown complete
[stage 2] extraction results: 11252 valid, 3770 failed validation

[stage 2] final tallies:
  folio_eval_dags.jsonl: 2189/2977 valid (73.5%)
    gemma3-12b: 179/713 valid (25.1%)
    llama3.2-3b: 561/735 valid (76.3%)
    mistral-7b-v0.3: 723/764 valid (94.6%)
    qwen2.5-7b: 726/765 valid (94.9%)
  musr_mm_eval_dags.jsonl: 2221/3954 valid (56.2%)
    gemma3-12b: 101/1000 valid (10.1%)
    llama3.2-3b: 580/954 valid (60.8%)
    mistral-7b-v0.3: 776/1000 valid (77.6%)
    qwen2.5-7b: 764/1000 valid (76.4%)
  musr_op_eval_dags.jsonl: 3755/4113 valid (91.3%)
    gemma3-12b: 897/1029 valid (87.2%)
    llama3.2-3b: 923/1024 valid (90.1%)
    mistral-7b-v0.3: 987/1030 valid (95.8%)
    qwen2.5-7b: 948/1030 valid (92.0%)
  musr_ta_eval_dags.jsonl: 3087/3978 valid (77.6%)
    gem

In [2]:
# ============================================================================
# STAGE 2.1 -- POST-PROCESS: append <answer>-tag Answer node to each DAG
#
# STANDALONE. Reads the Stage 2 JSONL files and the Stage 1 eval CSVs, joins
# them on (dataset, question_id, model, sample_idx), and writes new JSONL
# files with one extra "Answer" node per DAG whose text is the trace's
# predicted_label from the CSV.
#
# Input :  <DAG_OUTPUT_DIR>/*_eval_dags.jsonl       (from Stage 2)
#          <STAGE1_OUTPUT_DIR>/*_eval_cots.csv      (from Stage 1)
# Output:  <DAG_OUTPUT_DIR>/*_eval_dags_with_answer.jsonl  (new files alongside)
#
# Per-record behavior:
#
#   parse_ok=True  + Conclusion found + CSV match
#     -> append { id: aN, type: "Answer", text: predicted_label,
#                 support: [[conclusion_id]], defeats: [], gate: "And" }
#        Set answer_node_id = aN.
#
#   parse_ok=False + Conclusion found + CSV match
#     -> SAME append behavior, but the Answer node also carries
#        "from_failed_dag": True so downstream consumers can filter it
#        out (or treat it differently) when scoring ensemble agreement.
#        Set answer_node_id = aN and answer_from_failed_dag = True.
#
#   no Conclusion in the DAG, OR no CSV match, OR dag is None
#     -> record forwarded unchanged with answer_node_id = null and a
#        postproc_errors entry explaining why.
# ============================================================================

import csv, glob, json, os
from typing import Dict, Optional, Tuple

# ----------------------------------------------------------------------------
# Paths -- adjust to match your setup.
# ----------------------------------------------------------------------------
STAGE1_OUTPUT_DIR = "./outputs_2234"
DAG_OUTPUT_DIR    = "./outputs_2234"

JSONL_GLOB = os.path.join(DAG_OUTPUT_DIR, "*_eval_dags.jsonl")
# Skip files we ourselves produced, in case this script is re-run.
JSONL_SUFFIX_TO_SKIP = "_with_answer.jsonl"


# ----------------------------------------------------------------------------
# Build a lookup table: (dataset, question_id, model, sample_idx) -> predicted_label
# from every CSV under STAGE1_OUTPUT_DIR. Keying on dataset too in case any
# question_ids collide across datasets (they shouldn't, but defensive).
# ----------------------------------------------------------------------------
def load_csv_answers() -> Dict[Tuple[str, str, str, int], str]:
    answers: Dict[Tuple[str, str, str, int], str] = {}
    csv_paths = sorted(glob.glob(os.path.join(STAGE1_OUTPUT_DIR, "*_eval_cots.csv")))
    if not csv_paths:
        raise RuntimeError(f"No eval CSVs under {STAGE1_OUTPUT_DIR}")
    print(f"[postproc] loading answers from {len(csv_paths)} CSVs:")

    for csv_path in csv_paths:
        with open(csv_path, "r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            n_rows = n_kept = 0
            for row in reader:
                n_rows += 1
                qid = (row.get("question_id") or "").strip()
                ds  = (row.get("dataset") or "").strip()
                mdl = (row.get("model") or "").strip()
                try:
                    samp = int(row.get("sample_idx") or "")
                except ValueError:
                    continue
                ans = (row.get("predicted_label") or "").strip()
                if not (qid and mdl and ans):
                    continue
                answers[(ds, qid, mdl, samp)] = ans
                n_kept += 1
        print(f"  {os.path.basename(csv_path)}: {n_kept}/{n_rows} rows with answers")
    print(f"[postproc] total CSV answer-rows in lookup table: {len(answers)}")
    return answers


# ----------------------------------------------------------------------------
# Append an Answer node to the dag dict. Returns the new node's id, or None if
# we couldn't locate a Conclusion to anchor to. If from_failed_dag=True, the
# Answer node carries that marker so downstream code can identify Answer nodes
# that came from a DAG that failed Stage 2 validation.
# ----------------------------------------------------------------------------
def append_answer_node(dag: Dict, predicted_label: str,
                       from_failed_dag: bool = False) -> Optional[str]:
    if not isinstance(dag, dict) or "nodes" not in dag:
        return None
    nodes = dag["nodes"]
    if not isinstance(nodes, list):
        return None

    # Find the (first) Conclusion node and collect existing ids
    concl_id: Optional[str] = None
    existing_ids = set()
    for n in nodes:
        if isinstance(n, dict):
            nid = n.get("id")
            if isinstance(nid, str):
                existing_ids.add(nid)
            if n.get("type") == "Conclusion" and concl_id is None and isinstance(nid, str):
                concl_id = nid
    if concl_id is None:
        return None

    # Pick a fresh id aN that doesn't collide
    i = 1
    while True:
        cand = f"a{i}"
        if cand not in existing_ids:
            break
        i += 1
    answer_id = cand

    answer_node = {
        "id":      answer_id,
        "type":    "Answer",
        "text":    predicted_label,
        "support": [[concl_id]],     # AND-bundle of size 1: Conclusion alone
        "defeats": [],
        "gate":    "And",            # |support| == 1
    }
    if from_failed_dag:
        answer_node["from_failed_dag"] = True

    nodes.append(answer_node)
    return answer_id


# ----------------------------------------------------------------------------
# Main: stream-process each JSONL, lookup the matching answer, append, write
# to a sibling *_with_answer.jsonl file.
# ----------------------------------------------------------------------------
answers = load_csv_answers()

jsonl_paths = sorted(p for p in glob.glob(JSONL_GLOB)
                     if not p.endswith(JSONL_SUFFIX_TO_SKIP))
print(f"\n[postproc] processing {len(jsonl_paths)} DAG JSONLs:")
for p in jsonl_paths:
    print(f"  {p}")
if not jsonl_paths:
    raise RuntimeError(f"No DAG JSONLs at {JSONL_GLOB}")

global_counters = {
    "records": 0,
    "appended_ok": 0,            # parse_ok=True, Answer appended
    "appended_failed": 0,         # parse_ok=False, Answer still appended (Conclusion found)
    "skipped_no_match": 0,       # no matching CSV row
    "skipped_no_conclusion": 0,  # DAG has no Conclusion node to anchor to
    "skipped_no_dag": 0,         # dag is None / not a dict
}

for jsonl_path in jsonl_paths:
    out_path = jsonl_path.replace(".jsonl", "_with_answer.jsonl")

    n_records = 0
    n_appended_ok = n_appended_failed = 0
    n_skipped_no_match = n_skipped_no_concl = n_skipped_no_dag = 0
    n_csv_label_disagrees = 0

    with open(jsonl_path, "r", encoding="utf-8") as fin, \
         open(out_path,    "w", encoding="utf-8") as fout:
        for line in fin:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                # Forward unparseable lines verbatim so we don't silently lose data
                fout.write(line if line.endswith("\n") else line + "\n")
                continue

            n_records += 1
            rec["answer_node_id"] = None
            rec.setdefault("postproc_errors", [])

            # Look up the CSV answer for this exact (dataset, qid, model, sample_idx).
            try:
                samp = int(rec.get("sample_idx", -1))
            except (TypeError, ValueError):
                samp = -1
            key = (
                rec.get("dataset", ""),
                rec.get("question_id", ""),
                rec.get("model", ""),
                samp,
            )
            csv_answer = answers.get(key)
            if csv_answer is None:
                n_skipped_no_match += 1
                rec["postproc_errors"].append(f"no CSV row for key {key}")
                fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
                continue

            # If predicted_label in JSONL disagrees with the CSV, log it. We
            # still use the CSV value (the user's source of truth), but flag
            # the discrepancy so any drift between Stage 1 and Stage 2 is visible.
            if (rec.get("predicted_label") or "").strip() != csv_answer:
                n_csv_label_disagrees += 1
                rec["postproc_errors"].append(
                    f"predicted_label mismatch: jsonl={rec.get('predicted_label')!r} "
                    f"csv={csv_answer!r} (using CSV value)")

            # Whether we append depends on whether there's a usable DAG with a
            # Conclusion node, NOT on parse_ok. Failed-validation DAGs still
            # get an Answer node when they have a Conclusion -- we just mark it.
            if not isinstance(rec.get("dag"), dict):
                n_skipped_no_dag += 1
                rec["postproc_errors"].append("dag is null/missing")
                fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
                continue

            from_failed = not bool(rec.get("parse_ok"))
            answer_node_id = append_answer_node(
                rec["dag"], csv_answer, from_failed_dag=from_failed
            )
            if answer_node_id is None:
                n_skipped_no_concl += 1
                rec["postproc_errors"].append(
                    "could not locate Conclusion node in DAG")
            else:
                rec["answer_node_id"] = answer_node_id
                rec["answer_from_failed_dag"] = from_failed
                if from_failed:
                    n_appended_failed += 1
                else:
                    n_appended_ok += 1

            fout.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"\n  {os.path.basename(jsonl_path)} -> {os.path.basename(out_path)}")
    print(f"    records:              {n_records}")
    print(f"    appended (parse_ok):  {n_appended_ok}")
    print(f"    appended (failed):    {n_appended_failed}")
    print(f"    skipped (no CSV):     {n_skipped_no_match}")
    print(f"    skipped (no Concl):   {n_skipped_no_concl}")
    print(f"    skipped (no DAG):     {n_skipped_no_dag}")
    print(f"    jsonl/csv mismatch:   {n_csv_label_disagrees}")

    global_counters["records"]               += n_records
    global_counters["appended_ok"]           += n_appended_ok
    global_counters["appended_failed"]       += n_appended_failed
    global_counters["skipped_no_match"]      += n_skipped_no_match
    global_counters["skipped_no_conclusion"] += n_skipped_no_concl
    global_counters["skipped_no_dag"]        += n_skipped_no_dag

print("\n[postproc] global totals:")
for k, v in global_counters.items():
    print(f"  {k}: {v}")
print(f"\n[postproc] new files written to {DAG_OUTPUT_DIR}/*_with_answer.jsonl")

[postproc] loading answers from 4 CSVs:
  folio_eval_cots.csv: 2977/3060 rows with answers
  musr_mm_eval_cots.csv: 3954/4000 rows with answers
  musr_op_eval_cots.csv: 4113/4120 rows with answers
  musr_ta_eval_cots.csv: 3978/4000 rows with answers
[postproc] total CSV answer-rows in lookup table: 15022

[postproc] processing 4 DAG JSONLs:
  ./outputs_2234/folio_eval_dags.jsonl
  ./outputs_2234/musr_mm_eval_dags.jsonl
  ./outputs_2234/musr_op_eval_dags.jsonl
  ./outputs_2234/musr_ta_eval_dags.jsonl



  folio_eval_dags.jsonl -> folio_eval_dags_with_answer.jsonl
    records:              2977
    appended (parse_ok):  2189
    appended (failed):    785
    skipped (no CSV):     0
    skipped (no Concl):   0
    skipped (no DAG):     3
    jsonl/csv mismatch:   0

  musr_mm_eval_dags.jsonl -> musr_mm_eval_dags_with_answer.jsonl
    records:              3954
    appended (parse_ok):  2221
    appended (failed):    1410
    skipped (no CSV):     0
    skipped (no Concl):   0
    skipped (no DAG):     323
    jsonl/csv mismatch:   0

  musr_op_eval_dags.jsonl -> musr_op_eval_dags_with_answer.jsonl
    records:              4113
    appended (parse_ok):  3755
    appended (failed):    352
    skipped (no CSV):     0
    skipped (no Concl):   0
    skipped (no DAG):     6
    jsonl/csv mismatch:   0

  musr_ta_eval_dags.jsonl -> musr_ta_eval_dags_with_answer.jsonl
    records:              3978
    appended (parse_ok):  3087
    appended (failed):    837
    skipped (no CSV):     0
    s